# 🔍 Vérification IAM Role et Permissions S3

Notebook pour diagnostiquer les problèmes d'accès S3 et vérifier la configuration IAM.


In [ ]:
# CELLULE 1: Vérifier les Credentials AWS et IAM Role
import boto3
import json
import os
from pathlib import Path

print("="*70)
print("🔍 VÉRIFICATION CREDENTIALS AWS ET IAM ROLE")
print("="*70)
print()

# 1. Vérifier comment boto3 trouve les credentials
print("📋 MÉTHODE DE RECHERCHE DES CREDENTIALS:")
print("-" * 70)

try:
    session = boto3.Session()
    credentials = session.get_credentials()
    
    if credentials:
        print(f"✅ Credentials trouvés")
        print(f"   Méthode: {type(credentials).__name__}")
        
        # Vérifier si c'est un IAM role (dans SageMaker)
        if 'role' in str(credentials).lower() or 'assume' in str(credentials).lower():
            print(f"   ✅ IAM Role détecté (SageMaker Studio)")
        else:
            print(f"   ℹ️ Credentials standards (clés d'accès)")
        
        # Afficher l'Access Key (partiellement masquée)
        access_key = credentials.access_key
        if access_key:
            masked_key = access_key[:8] + "*" * (len(access_key) - 12) + access_key[-4:]
            print(f"   Access Key: {masked_key}")
    else:
        print("❌ Aucun credential trouvé")
        print("   Vérifiez votre configuration AWS")
        
except Exception as e:
    print(f"❌ Erreur lors de la vérification: {e}")

print()

# 2. Vérifier les variables d'environnement
print("📋 VARIABLES D'ENVIRONNEMENT:")
print("-" * 70)

env_vars = [
    'AWS_ACCESS_KEY_ID',
    'AWS_SECRET_ACCESS_KEY',
    'AWS_SESSION_TOKEN',  # Pour les roles temporaires
    'AWS_REGION',
    'AWS_DEFAULT_REGION',
    'AWS_PROFILE',
    'S3_BUCKET_NAME'
]

for var in env_vars:
    value = os.environ.get(var)
    if value:
        if 'KEY' in var or 'SECRET' in var or 'TOKEN' in var:
            masked = value[:8] + "*" * (len(value) - 12) + value[-4:] if len(value) > 12 else "***"
            print(f"   ✅ {var}: {masked}")
        else:
            print(f"   ✅ {var}: {value}")
    else:
        print(f"   ❌ {var}: non défini")

print()

# 3. Vérifier si on est dans SageMaker
print("📋 ENVIRONNEMENT:")
print("-" * 70)

try:
    from sagemaker_studio import Project
    project = Project()
    print(f"   ✅ SageMaker Studio détecté")
    print(f"   IAM Role du projet: {getattr(project, 'iam_role', 'N/A')}")
    print(f"   S3 Root: {project.s3.root if hasattr(project, 's3') else 'N/A'}")
except ImportError:
    print(f"   ℹ️ Pas dans SageMaker Studio (exécution locale)")
except Exception as e:
    print(f"   ⚠️ Erreur SageMaker: {e}")

print()


## 🔄 Rafraîchir les Credentials Expirés

Si vous obtenez une erreur `ExpiredToken`, utilisez la cellule ci-dessous pour rafraîchir vos credentials AWS.


In [ ]:
# CELLULE 1.5: Rafraîchir les Credentials AWS Expirés
import boto3
import os
import subprocess
import sys
from botocore.exceptions import ClientError

print("="*70)
print("🔄 RAFRAÎCHISSEMENT DES CREDENTIALS AWS")
print("="*70)
print()

# Méthode 1: Vérifier si on utilise AWS SSO
def refresh_aws_sso():
    """Rafraîchir via AWS SSO"""
    import platform
    
    try:
        print("🔄 Tentative de rafraîchissement via AWS SSO...")
        
        # Sur Windows, utiliser shell=True
        shell_needed = platform.system() == 'Windows'
        
        result = subprocess.run(
            ['aws', 'sso', 'login'],
            capture_output=True,
            text=True,
            timeout=120,
            shell=shell_needed
        )
        
        if result.returncode == 0:
            print("✅ AWS SSO login réussi!")
            # Les credentials SSO sont automatiquement chargés par boto3 depuis ~/.aws/config
            print("✅ Credentials SSO rechargés")
            print("   ℹ️ Note: Vous devrez peut-être redémarrer le kernel pour charger les nouveaux credentials")
            return True
        else:
            print(f"⚠️ AWS SSO login échoué")
            if result.stderr:
                print(f"   Erreur: {result.stderr[:200]}")
            return False
    except FileNotFoundError:
        print("⚠️ AWS CLI non installé ou non dans le PATH")
        print("   → Installez AWS CLI: https://aws.amazon.com/cli/")
        return False
    except subprocess.TimeoutExpired:
        print("⚠️ Timeout lors du login AWS SSO")
        print("   → Essayez d'exécuter 'aws sso login' manuellement dans un terminal")
        return False
    except Exception as e:
        print(f"⚠️ Erreur lors du refresh SSO: {e}")
        return False

# Méthode 2: Vérifier si le token est expiré et donner des instructions
def check_token_expired():
    """Vérifier si le token est expiré"""
    try:
        sts = boto3.client('sts')
        identity = sts.get_caller_identity()
        print("✅ Le token actuel est valide!")
        print(f"   Account: {identity.get('Account', 'N/A')}")
        print(f"   ARN: {identity.get('Arn', 'N/A')}")
        return False
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'ExpiredToken' or 'expired' in str(e).lower():
            print("❌ Token expiré détecté!")
            return True
        else:
            print(f"⚠️ Erreur inattendue: {e}")
            return False
    except Exception as e:
        print(f"⚠️ Erreur lors de la vérification: {e}")
        return False

# Vérifier la source des credentials
def detect_credentials_source():
    """Détecter d'où viennent les credentials"""
    if os.environ.get('AWS_SESSION_TOKEN'):
        return 'env_with_token'  # Credentials temporaires depuis env
    elif os.environ.get('AWS_ACCESS_KEY_ID'):
        return 'env'  # Credentials permanents depuis env
    elif os.environ.get('AWS_PROFILE'):
        return 'profile'  # Profile AWS
    else:
        # Vérifier si boto3 trouve des credentials
        try:
            session = boto3.Session()
            credentials = session.get_credentials()
            if credentials:
                # Vérifier si c'est depuis un fichier
                import os.path
                aws_config = os.path.expanduser('~/.aws/config')
                aws_creds = os.path.expanduser('~/.aws/credentials')
                if os.path.exists(aws_config) or os.path.exists(aws_creds):
                    return 'aws_config'
        except:
            pass
        return 'unknown'

# Vérifier l'état actuel
is_expired = check_token_expired()
cred_source = detect_credentials_source()

if is_expired:
    print()
    print("💡 SOLUTIONS POUR RAFRAÎCHIR LE TOKEN:")
    print("-" * 70)
    print()
    
    # Instructions spécifiques selon la source des credentials
    if cred_source == 'env_with_token':
        print("🔍 DÉTECTÉ: Credentials temporaires depuis variables d'environnement")
        print()
        print("="*70)
        print("📋 OPTION 1: CONFIGURER AWS SSO (Recommandé)")
        print("="*70)
        print()
        print("Étape 1: Obtenir les informations SSO de votre organisation AWS")
        print("   → Contactez votre administrateur AWS ou consultez:")
        print("      - AWS Console > IAM Identity Center")
        print("      - L'URL SSO (ex: https://d-1234567890.awsapps.com/start)")
        print("      - La région SSO (ex: us-east-1)")
        print()
        print("Étape 2: Configurer AWS SSO dans PowerShell")
        print("   → Exécutez:")
        print("      aws configure sso")
        print()
        print("   → Répondez aux questions:")
        print("      SSO start URL: [votre-url-sso]")
        print("      SSO region: [votre-region-sso]")
        print("      SSO account ID: [votre-account-id]")
        print("      SSO role name: [votre-role, ex: AdministratorAccess]")
        print("      CLI profile name: [ex: default]")
        print()
        print("Étape 3: Se connecter")
        print("   → Exécutez:")
        print("      aws sso login --profile default")
        print()
        print("Étape 4: Utiliser le profile dans PowerShell")
        print("   → $env:AWS_PROFILE = 'default'")
        print("   → Redémarrez le kernel Jupyter")
        print()
        print("="*70)
        print("📋 OPTION 2: NOUVEAUX CREDENTIALS SANS SSO (Solution rapide)")
        print("="*70)
        print()
        print("Si AWS SSO n'est pas disponible, obtenez de nouveaux credentials:")
        print()
        print("Méthode A: Depuis AWS Console (pour credentials permanents)")
        print("   1. Allez dans AWS Console > IAM > Users > Votre utilisateur")
        print("   2. Security credentials > Create access key")
        print("   3. Copiez l'Access Key ID et Secret Access Key")
        print("   4. Dans PowerShell, configurez:")
        print("      $env:AWS_ACCESS_KEY_ID = 'AKIA...'")
        print("      $env:AWS_SECRET_ACCESS_KEY = 'votre-secret'")
        print("      $env:AWS_SESSION_TOKEN = ''  # Vide pour credentials permanents")
        print("      $env:AWS_REGION = 'us-west-2'")
        print()
        print("Méthode B: Via fichier .env (Recommandé pour le projet)")
        print("   1. Créez/modifiez .env à la racine du projet:")
        print("      AWS_ACCESS_KEY_ID=votre-key")
        print("      AWS_SECRET_ACCESS_KEY=votre-secret")
        print("      AWS_REGION=us-west-2")
        print("   2. Redémarrez le kernel Jupyter")
        print()
        print("Méthode C: Script/processus existant")
        print("   → Si vous avez un script qui génère les credentials temporaires")
        print("   → Ré-exécutez-le et copiez les valeurs dans PowerShell")
        print()
    else:
        print("OPTION 1: AWS SSO (recommandé)")
        print("   → Exécutez dans un terminal:")
        print("      aws sso login")
        print("   → Puis configurez le profile ou exportez les credentials")
        print()
        print("OPTION 2: Si vous utilisez AWS CLI avec un profile")
        print("   → Exécutez:")
        print("      aws configure list")
        print("   → Si nécessaire, reconfigurez:")
        print("      aws configure")
        print()
        print("OPTION 3: Si vous avez obtenu les credentials via un script")
        print("   → Ré-exécutez le script qui génère les credentials")
        print("   → Mettez à jour les variables d'environnement")
        print()
        print("OPTION 4: Obtenir de nouveaux credentials depuis AWS Console")
        print("   → Allez dans IAM > Users > Votre utilisateur")
        print("   → Security credentials > Create access key")
        print("   → Configurez-les dans votre .env ou variables d'environnement")
        print()
    
    # Afficher les commandes PowerShell pour obtenir de nouveaux credentials
    if cred_source == 'env_with_token':
        print("📋 COMMANDES POWERSHELL POUR RAFRAÎCHIR:")
        print("-" * 70)
        print()
        print("# Méthode 1: Via AWS SSO")
        print("aws sso login")
        print("# Puis obtenir les credentials temporaires:")
        print('$cred = aws configure export-credentials --format env')
        print('# Ou manuellement après aws sso login, utilisez:')
        print('$env:AWS_PROFILE = "default"  # ou votre profile SSO')
        print()
        print("# Méthode 2: Via script existant")
        print("# Si vous avez un script qui génère les credentials,")
        print("# ré-exécutez-le et copiez les valeurs dans PowerShell:")
        print('$env:AWS_ACCESS_KEY_ID = "ASIA..."')
        print('$env:AWS_SECRET_ACCESS_KEY = "votre-secret"')
        print('$env:AWS_SESSION_TOKEN = "IQoJb..."')
        print('$env:AWS_REGION = "us-west-2"')
        print()
        print("⚠️ IMPORTANT: Après avoir mis à jour les variables d'environnement,")
        print("   redémarrez le kernel Jupyter pour que les changements prennent effet.")
        print()
    
    # Essayer automatiquement AWS SSO si disponible
    print("🔄 Tentative automatique de rafraîchissement via AWS SSO...")
    sso_success = refresh_aws_sso()
    
    if not sso_success and cred_source == 'env_with_token':
        print()
        print("💡 SOLUTION RAPIDE POUR POWERSHELL:")
        print("-" * 70)
        print("1. Ouvrez un nouveau terminal PowerShell")
        print("2. Exécutez: aws sso login")
        print("3. Ensuite, dans le notebook, redémarrez le kernel:")
        print("   Kernel > Restart Kernel")
        print("4. Ré-exécutez cette cellule pour vérifier")
        print()
    
    # Re-vérifier après tentative de refresh
    print()
    print("🔍 Vérification après rafraîchissement...")
    if not check_token_expired():
        print()
        print("✅ SUCCÈS! Le token a été rafraîchi.")
        print("   Vous pouvez maintenant ré-exécuter les cellules de test S3.")
        print()
        print("💡 Si vous avez mis à jour les variables d'environnement dans PowerShell,")
        print("   redémarrez le kernel Jupyter (Kernel > Restart Kernel)")
    else:
        print()
        print("⚠️ Le token est toujours expiré.")
        if cred_source == 'env_with_token':
            print()
            print("🔧 ACTION REQUISE:")
            print("   1. Dans PowerShell, exécutez: aws sso login")
            print("   2. Ou ré-obtenez les credentials depuis votre source habituelle")
            print("   3. Redémarrez le kernel Jupyter")
            print("   4. Ré-exécutez cette cellule")
        else:
            print("   Veuillez suivre une des options ci-dessus manuellement.")
else:
    print()
    print("ℹ️ Vos credentials sont valides, aucune action nécessaire.")
    print(f"   Source détectée: {cred_source}")

print()


## 💡 Script PowerShell pour rafraîchir automatiquement

Copiez et exécutez ce script dans PowerShell pour obtenir de nouveaux credentials après `aws sso login`.


In [ ]:
# GUIDE COMPLET: CONFIGURATION AWS
print("="*70)
print("📝 GUIDE COMPLET - CONFIGURATION AWS")
print("="*70)
print()
print()
print("="*70)
print("🔧 GUIDE 1: CONFIGURER AWS SSO (Interactif)")
print("="*70)
print()
print("Dans PowerShell, exécutez:")
print("   aws configure sso")
print()
print("Répondez aux questions suivantes:")
print()
print("   ? SSO start URL: [exemple: https://d-1234567890.awsapps.com/start]")
print("     → Obtenez cette URL depuis AWS Console > IAM Identity Center")
print("     → Ou demandez à votre administrateur AWS")
print()
print("   ? SSO region: [exemple: us-east-1 ou us-west-2]")
print("     → La région où votre SSO est configuré")
print()
print("   ? SSO account ID: [exemple: 123456789012]")
print("     → Votre Account ID AWS (visible en haut à droite dans AWS Console)")
print()
print("   ? SSO role name: [exemple: AdministratorAccess ou PowerUserAccess]")
print("     → Le rôle que vous voulez utiliser (listé après configuration)")
print()
print("   ? CLI profile name: [exemple: default]")
print("     → Le nom du profile (utilisez 'default' pour simplicité)")
print()
print("Ensuite, connectez-vous:")
print("   aws sso login --profile default")
print()
print("Et dans PowerShell, activez le profile:")
print('   $env:AWS_PROFILE = "default"')
print()
print("Redémarrez le kernel Jupyter et les credentials seront automatiquement utilisés!")
print()
print()
print("="*70)
print("🔧 GUIDE 2: OBTENIR DE NOUVEAUX CREDENTIALS (Sans SSO)")
print("="*70)
print()
print("Option A: Depuis AWS Console (Recommandé si pas de SSO)")
print("   1. Connectez-vous à AWS Console: https://console.aws.amazon.com/")
print("   2. Allez dans IAM > Users > [Votre nom d'utilisateur]")
print("   3. Onglet 'Security credentials'")
print("   4. Cliquez sur 'Create access key'")
print("   5. Choisissez 'Command Line Interface (CLI)' ou 'Application running outside AWS'")
print("   6. Copiez l'Access Key ID et Secret Access Key")
print("   7. ⚠️ IMPORTANT: Sauvegardez la Secret Key - elle ne sera plus visible!")
print()
print("Option B: Créer un fichier .env (Recommandé pour le projet)")
print("   1. Créez un fichier .env à la racine du projet:")
print("      C:\\Users\\rayya\\Desktop\\Datathon2025\\.env")
print()
print("   2. Ajoutez ces lignes:")
print("      AWS_ACCESS_KEY_ID=votre-access-key-ici")
print("      AWS_SECRET_ACCESS_KEY=votre-secret-key-ici")
print("      AWS_REGION=us-west-2")
print("      S3_BUCKET_NAME=csv-file-store-70f60a80")
print()
print("   3. Le fichier .env est ignoré par Git (sécurisé)")
print("   4. Redémarrez le kernel Jupyter")
print()
print("Option C: Configurer dans PowerShell (temporaire)")
print("   Dans PowerShell, exécutez:")
print('   $env:AWS_ACCESS_KEY_ID = "AKIA..."')
print('   $env:AWS_SECRET_ACCESS_KEY = "votre-secret-key"')
print('   $env:AWS_REGION = "us-west-2"')
print('   $env:AWS_SESSION_TOKEN = ""  # Vide pour credentials permanents')
print()
print("   ⚠️ Note: Ces variables ne persistent que pour la session PowerShell")
print("   → Utilisez .env pour une solution permanente")
print()
print()
print("="*70)
print("✅ APRÈS CONFIGURATION")
print("="*70)
print()
print("1. Redémarrez le kernel Jupyter (Kernel > Restart Kernel)")
print("2. Ré-exécutez la cellule de vérification ci-dessus (Cellule 1)")
print("3. Ré-exécutez les tests S3 (Cellule 2)")
print("4. Les tests devraient maintenant fonctionner! ✅")
print()


In [ ]:
# CELLULE 2: Tester les Permissions S3
import boto3
from botocore.exceptions import ClientError
import os

print("="*70)
print("🧪 TEST DES PERMISSIONS S3")
print("="*70)
print()

# Récupérer le bucket depuis .env ou variable d'environnement
try:
    from dotenv import load_dotenv
    load_dotenv('.env')
except:
    pass

bucket_name = os.environ.get('S3_BUCKET_NAME') or os.environ.get('AWS_S3_BUCKET')
region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

if not bucket_name:
    print("⚠️ Aucun bucket configuré")
    print("   Configurez S3_BUCKET_NAME dans .env ou variable d'environnement")
else:
    print(f"📦 Bucket à tester: {bucket_name}")
    print(f"🌍 Région: {region}")
    print()
    
    try:
        s3 = boto3.client('s3', region_name=region)
        
        # TEST 1: Vérifier que le bucket existe et est accessible
        print("TEST 1: Accès au bucket (head_bucket)...")
        try:
            s3.head_bucket(Bucket=bucket_name)
            print("   ✅ SUCCÈS - Le bucket existe et est accessible")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == '403':
                print(f"   ❌ ERREUR 403 - Permission refusée")
                print(f"      → Votre IAM role/user n'a pas la permission s3:ListBucket")
            elif error_code == '404':
                print(f"   ❌ ERREUR 404 - Bucket n'existe pas")
            else:
                print(f"   ❌ ERREUR: {error_code} - {e}")
        print()
        
        # TEST 2: Lister les objets (permission ListObjects)
        print("TEST 2: Liste des objets (list_objects_v2)...")
        try:
            response = s3.list_objects_v2(Bucket=bucket_name, MaxKeys=5)
            count = response.get('KeyCount', 0)
            print(f"   ✅ SUCCÈS - {count} objets trouvés")
            if 'Contents' in response:
                print(f"   Exemples:")
                for obj in response['Contents'][:3]:
                    print(f"      - {obj['Key']}")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == 'ExpiredToken':
                print(f"   ❌ ERREUR ExpiredToken - Le token de session a expiré")
                print(f"      → ⚠️ IMPORTANT: Utilisez la cellule 'Rafraîchir les Credentials Expirés' ci-dessus")
                print(f"      → Ou exécutez: aws sso login (si vous utilisez AWS SSO)")
            elif error_code == 'AccessDenied':
                print(f"   ❌ ERREUR AccessDenied")
                print(f"      → Permission manquante: s3:ListBucket")
            else:
                print(f"   ❌ ERREUR: {error_code} - {e}")
        print()
        
        # TEST 3: Lire un objet (permission GetObject)
        print("TEST 3: Lecture d'un objet (get_object)...")
        test_key = "data/raw/2025-08-15_composition_sp500.csv"
        try:
            obj = s3.get_object(Bucket=bucket_name, Key=test_key)
            print(f"   ✅ SUCCÈS - Objet '{test_key}' lisible")
            print(f"   Taille: {obj['ContentLength']} bytes")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == 'ExpiredToken':
                print(f"   ❌ ERREUR ExpiredToken - Le token de session a expiré")
                print(f"      → ⚠️ IMPORTANT: Utilisez la cellule 'Rafraîchir les Credentials Expirés' ci-dessus")
                print(f"      → Ou exécutez: aws sso login (si vous utilisez AWS SSO)")
            elif error_code == 'NoSuchKey':
                print(f"   ⚠️ NoSuchKey - L'objet '{test_key}' n'existe pas")
                print(f"      → Le bucket est accessible MAIS le fichier n'est pas là")
                print(f"      → Solution: Uploader le fichier vers S3")
            elif error_code == 'AccessDenied':
                print(f"   ❌ ERREUR AccessDenied")
                print(f"      → Permission manquante: s3:GetObject")
            else:
                print(f"   ❌ ERREUR: {error_code} - {e}")
        print()
        
        # TEST 4: Écrire un objet (permission PutObject)
        print("TEST 4: Écriture d'un objet (put_object)...")
        try:
            test_write_key = "reguai-test-permissions.txt"
            s3.put_object(Bucket=bucket_name, Key=test_write_key, Body=b"test")
            print(f"   ✅ SUCCÈS - Écriture réussie")
            
            # Nettoyer
            s3.delete_object(Bucket=bucket_name, Key=test_write_key)
            print(f"   ✅ Objet test supprimé")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code == 'ExpiredToken':
                print(f"   ❌ ERREUR ExpiredToken - Le token de session a expiré")
                print(f"      → ⚠️ IMPORTANT: Utilisez la cellule 'Rafraîchir les Credentials Expirés' ci-dessus")
                print(f"      → Ou exécutez: aws sso login (si vous utilisez AWS SSO)")
            elif error_code == 'AccessDenied':
                print(f"   ❌ ERREUR AccessDenied")
                print(f"      → Permission manquante: s3:PutObject")
            else:
                print(f"   ❌ ERREUR: {error_code} - {e}")
        print()
        
    except Exception as e:
        print(f"❌ Erreur lors des tests: {e}")
        import traceback
        traceback.print_exc()


In [ ]:
# CELLULE 3: Vérifier l'IAM Role et ses Permissions
import boto3
from botocore.exceptions import ClientError
import json

print("="*70)
print("🔐 VÉRIFICATION IAM ROLE ET PERMISSIONS")
print("="*70)
print()

try:
    # Obtenir l'identité AWS actuelle
    sts = boto3.client('sts')
    identity = sts.get_caller_identity()
    
    print("👤 IDENTITÉ AWS ACTUELLE:")
    print("-" * 70)
    print(f"   User ID: {identity.get('UserId', 'N/A')}")
    print(f"   Account: {identity.get('Account', 'N/A')}")
    
    # Vérifier si c'est un role ou un user
    arn = identity.get('Arn', '')
    print(f"   ARN: {arn}")
    
    if ':role/' in arn:
        role_name = arn.split(':role/')[-1].split('/')[0]
        print(f"   ✅ Type: IAM Role")
        print(f"   Nom du Role: {role_name}")
        print()
        print(f"   💡 Dans AWS Console:")
        print(f"      → Allez dans IAM > Roles > {role_name}")
        print(f"      → Vérifiez les Policies attachées")
        print(f"      → Assurez-vous que S3 permissions sont présentes")
        
    elif ':user/' in arn:
        print(f"   ✅ Type: IAM User")
        print()
        print(f"   💡 Dans AWS Console:")
        print(f"      → Allez dans IAM > Users > {arn.split('/')[-1]}")
        print(f"      → Vérifiez les Policies attachées")
        
    else:
        print(f"   ℹ️ Type: {arn}")
    
    print()
    
except Exception as e:
    print(f"❌ Erreur lors de la vérification IAM: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# CELLULE 4: Résumé et Recommandations
print("="*70)
print("📋 RÉSUMÉ ET RECOMMANDATIONS")
print("="*70)
print()

print("💡 INTERPRÉTATION DES RÉSULTATS:")
print("-" * 70)
print()
print("✅ Si tous les tests passent:")
print("   → Les permissions IAM sont correctes")
print("   → Le problème peut être que les fichiers ne sont pas dans S3")
print()
print("❌ Si erreur ExpiredToken:")
print("   → ⚠️ LE TOKEN DE SESSION AWS A EXPIRÉ")
print("   → Solution URGENTE: Utilisez la cellule 'Rafraîchir les Credentials Expirés'")
print("   → Ou exécutez dans un terminal: aws sso login")
print("   → Puis ré-exécutez les tests")
print()
print("❌ Si erreur 403 ou AccessDenied:")
print("   → Permissions IAM manquantes")
print("   → Solution: Ajouter ces permissions à votre IAM role/user:")
print("      - s3:ListBucket (pour lister)")
print("      - s3:GetObject (pour lire)")
print("      - s3:PutObject (pour écrire)")
print()
print("⚠️ Si erreur NoSuchKey:")
print("   → Le bucket est accessible MAIS le fichier n'existe pas")
print("   → Solution: Uploader les fichiers CSV vers S3")
print("      s3://bucket-name/data/raw/2025-08-15_composition_sp500.csv")
print()
print("📝 PROCHAINES ÉTAPES:")
print("-" * 70)
print("1. Si ExpiredToken:")
print("   → Exécutez la cellule 'Rafraîchir les Credentials Expirés'")
print("   → Puis ré-exécutez les tests S3")
print()
print("2. Si permissions OK mais fichiers manquants:")
print("   → Utiliser la cellule suivante pour uploader les CSV")
print()
print("3. Si permissions manquantes:")
print("   → Aller dans AWS Console > IAM > Roles")
print("   → Trouver votre role SageMaker")
print("   → Ajouter une policy avec permissions S3")


In [ ]:
# CELLULE 5 (OPTIONNELLE): Uploader les CSV vers S3
import boto3
from pathlib import Path
import os

print("="*70)
print("📤 UPLOAD DES FICHIERS CSV VERS S3")
print("="*70)
print()

# Détecter la racine du projet
current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent.parent
else:
    project_root = current_dir

# Charger .env
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / '.env')
except:
    pass

bucket_name = os.environ.get('S3_BUCKET_NAME') or os.environ.get('AWS_S3_BUCKET')
region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

if not bucket_name:
    print("❌ Aucun bucket configuré")
else:
    # Fichiers à uploader
    files_to_upload = [
        ('data/raw/2025-08-15_composition_sp500.csv', 'data/raw/2025-08-15_composition_sp500.csv'),
        ('data/raw/2025-09-26_stocks-performance.csv', 'data/raw/2025-09-26_stocks-performance.csv')
    ]
    
    s3 = boto3.client('s3', region_name=region)
    
    for local_path, s3_key in files_to_upload:
        file_path = project_root / local_path
        
        if file_path.exists():
            try:
                print(f"📤 Upload: {local_path} → s3://{bucket_name}/{s3_key}")
                s3.upload_file(str(file_path), bucket_name, s3_key)
                print(f"   ✅ Succès!")
            except Exception as e:
                print(f"   ❌ Erreur: {e}")
        else:
            print(f"   ⚠️ Fichier non trouvé: {file_path}")
    
    print()
    print("✅ Upload terminé! Les fichiers sont maintenant dans S3.")
